<a href="https://colab.research.google.com/github/ruchira0011/Electricity-Demand-Forecasting-in-Great-Britain/blob/main/Uk_Energy_ARIMA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importing Data

In [ ]:
import pandas as pd

# List of NESO Historic Demand CSV URLs (you can extend this list)
urls = {
    2026: "https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/8a4a771c-3929-4e56-93ad-cdf13219dea5/download/demanddata_2026.csv",
    2025: "https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/b2bde559-3455-4021-b179-dfe60c0337b0/download/demanddata_2025.csv",
    2024: "https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/f6d02c0f-957b-48cb-82ee-09003f2ba759/download/demanddata_2024.csv",
    2023: "https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/bf5ab335-9b40-4ea4-b93a-ab4af7bce003/download/demanddata_2023.csv",
    2022: "https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/bb44a1b5-75b1-4db2-8491-257f23385006/download/demanddata_2022.csv",
    2021: "https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/18c69c42-f20d-46f0-84e9-e279045befc6/download/demanddata_2021.csv",
    2020: "https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/33ba6857-2a55-479f-9308-e5c4c53d4381/download/demanddata_2020.csv",
    2019: "https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/dd9de980-d724-415a-b344-d8ae11321432/download/demanddata_2019.csv",
    2018: "https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/fcb12133-0db0-4f27-a4a5-1669fd9f6d33/download/demanddata_2018.csv",
    2017: "https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/2f0f75b8-39c5-46ff-a914-ae38088ed022/download/demanddata_2017.csv",
    2016: "https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/3bb75a28-ab44-4a0b-9b1c-9be9715d3c44/download/demanddata_2016.csv",
    2015: "https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/cc505e45-65ae-4819-9b90-1fbb06880293/download/demanddata_2015.csv",
    2014: "https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/b9005225-49d3-40d1-921c-03ee2d83a2ff/download/demanddata_2014.csv",
    2013: "https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/2ff7aaff-8b42-4c1b-b234-9446573a1e27/download/demanddata_2013.csv",
    2012: "https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/4bf713a2-ea0c-44d3-a09a-63fc6a634b00/download/demanddata_2012.csv",
    2011: "https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/01522076-2691-4140-bfb8-c62284752efd/download/demanddata_2011.csv",
    2010: "https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/b3eae4a5-8c3c-4df1-b9de-7db243ac3a09/download/demanddata_2010.csv"
}

def load_neso_demand(urls: dict) -> pd.DataFrame:
    dfs = []
    for year, url in urls.items():
        print(f"Loading {year} data...")
        df_year = pd.read_csv(url)
        df_year["Year"] = year  # optional: keep track of source year
        dfs.append(df_year)
    return pd.concat(dfs, ignore_index=True)

df = load_neso_demand(urls)

print("Shape:", df.shape)
print(df.head())
print(df.columns)

Loading 2026 data...
Loading 2025 data...
Loading 2024 data...
Loading 2023 data...
Loading 2022 data...
Loading 2021 data...
Loading 2020 data...
Loading 2019 data...
Loading 2018 data...
Loading 2017 data...
Loading 2016 data...
Loading 2015 data...
Loading 2014 data...
Loading 2013 data...
Loading 2012 data...
Loading 2011 data...
Loading 2010 data...
Shape: (284782, 23)
  SETTLEMENT_DATE  SETTLEMENT_PERIOD     ND    TSD  ENGLAND_WALES_DEMAND  \
0      2026-01-01                  1  25107  29668                 23625   
1      2026-01-01                  2  25881  30361                 24341   
2      2026-01-01                  3  25355  30681                 23805   
3      2026-01-01                  4  24762  30225                 23226   
4      2026-01-01                  5  24111  29092                 22653   

   EMBEDDED_WIND_GENERATION  EMBEDDED_WIND_CAPACITY  \
0                      4252                    6606   
1                      4332                    6606   
2

# Data Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df.drop(columns=['SCOTTISH_TRANSFER','NSL_FLOW','ELECLINK_FLOW','VIKING_FLOW',
                 'GREENLINK_FLOW', 'IFA_FLOW', 'IFA2_FLOW', 'BRITNED_FLOW',
                 'MOYLE_FLOW', 'EAST_WEST_FLOW', 'NEMO_FLOW'], inplace=True)

# 1. Convert date column to datetime
df["SETTLEMENT_DATE"] = pd.to_datetime(df["SETTLEMENT_DATE"], format='mixed',
                                       dayfirst=True)

# 2. Ensure settlement period is numeric
df["SETTLEMENT_PERIOD"] = pd.to_numeric(df["SETTLEMENT_PERIOD"])

# 3. Create full timestamp (each period = 30 minutes)
df["timestamp"] = df["SETTLEMENT_DATE"] + pd.to_timedelta(
    (df["SETTLEMENT_PERIOD"] - 1) * 30, unit="min"
)

# 4. Set timestamp as index
df = df.set_index("timestamp")

# 5. Sort by time
df = df.sort_index()

# 6. Assign demand column
demand_col = "ND"
demand = df[demand_col]

# 7. Basic dataset info
print("Data shape:", df.shape)
print("Date range:", df.index.min(), "to", df.index.max())

# 8. Demand statistics
print(demand.describe())


# missing value handling
# Missingness
missing = df.isna().mean().sort_values(ascending=False) * 100
print("Top missing % columns:")
display(missing.head(10))

# Duplicates in time index
dup_count = df.index.duplicated().sum()
print("Duplicate timestamps:", dup_count)

# If duplicates exist, aggregate them
if dup_count > 0:
    df = df.groupby(df.index).mean(numeric_only=True)
    print("After aggregating duplicates -> shape:", df.shape)


Data shape: (284782, 12)
Date range: 2010-01-01 00:00:00 to 2026-03-30 23:30:00
count    284782.000000
mean      30569.080419
std        7696.802738
min       12803.000000
25%       24410.000000
50%       29769.500000
75%       36140.000000
max       59095.000000
Name: ND, dtype: float64
Top missing % columns:


,0
SETTLEMENT_DATE,0.0
SETTLEMENT_PERIOD,0.0
ND,0.0
TSD,0.0
ENGLAND_WALES_DEMAND,0.0
EMBEDDED_WIND_GENERATION,0.0
EMBEDDED_WIND_CAPACITY,0.0
EMBEDDED_SOLAR_GENERATION,0.0
EMBEDDED_SOLAR_CAPACITY,0.0
NON_BM_STOR,0.0


Duplicate timestamps: 32
After aggregating duplicates -> shape: (284750, 11)


In [ ]:
from statsmodels.tsa.arima.model import ARIMA
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------
# Train / Test split
# -------------------------------
train_ts = series.loc[:'2023']
test_ts  = series.loc['2024':]

# -------------------------------
# Fit ARIMA
# -------------------------------
model = ARIMA(train_ts, order=(2,1,2))
model_fit = model.fit()

# -------------------------------
# FORECAST HORIZONS
# -------------------------------
# 3-hour data:
# Day-ahead = 8 steps
# Week-ahead = 56 steps

forecast_day = model_fit.forecast(steps=8)
forecast_week = model_fit.forecast(steps=56)

# -------------------------------
# ACTUAL VALUES
# -------------------------------
actual_day = test_ts[:8]
actual_week = test_ts[:56]

# -------------------------------
# Evaluation
# -------------------------------
mae_day = np.mean(np.abs(actual_day.values - forecast_day.values))
mae_week = np.mean(np.abs(actual_week.values - forecast_week.values))

rmse_day = np.sqrt(np.mean((actual_day.values - forecast_day.values)**2))
rmse_week = np.sqrt(np.mean((actual_week.values - forecast_week.values)**2))

print("ARIMA Day-ahead MAE:", mae_day)
print("ARIMA Week-ahead MAE:", mae_week)

# -------------------------------
# Plot Day-ahead
# -------------------------------
plt.figure(figsize=(12,4))
plt.plot(actual_day.values, label='Actual')
plt.plot(forecast_day.values, label='Forecast')
plt.title("ARIMA Day-ahead Forecast")
plt.legend()
plt.show()

# -------------------------------
# Plot Week-ahead
# -------------------------------
plt.figure(figsize=(12,4))
plt.plot(actual_week.values, label='Actual')
plt.plot(forecast_week.values, label='Forecast')
plt.title("ARIMA Week-ahead Forecast")
plt.legend()
plt.show()

In [ ]:
import itertools
from statsmodels.tsa.arima.model import ARIMA
import numpy as np

p = [1,2]
d = [1]
q = [1,2]

best_mae = float("inf")
best_order = None

# 3-hour data
day_steps = 8   # 24h
week_steps = 56 # 7 days

for order in itertools.product(p,d,q):
    try:
        model = ARIMA(train_ts, order=order)
        model_fit = model.fit()

        # ---- Day-ahead forecast ----
        forecast_day = model_fit.forecast(steps=day_steps)
        actual_day = test_ts[:day_steps]

        mae_day = np.mean(np.abs(actual_day.values - forecast_day.values))

        # ---- Week-ahead forecast ----
        forecast_week = model_fit.forecast(steps=week_steps)
        actual_week = test_ts[:week_steps]

        mae_week = np.mean(np.abs(actual_week.values - forecast_week.values))

        # Combine both (simple average)
        mae = (mae_day + mae_week) / 2

        if mae < best_mae:
            best_mae = mae
            best_order = order

    except:
        continue

print("Best ARIMA:", best_order, "MAE:", best_mae)

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


Best ARIMA: (2, 1, 2) MAE: 3639.129592333853


In [ ]:
import pandas as pd

df_results = pd.DataFrame(results, columns=["config", "MAE"])
df_results = df_results.sort_values("MAE")

print(df_results)